<a href="https://colab.research.google.com/github/marcinsawinski/scikit-learn-mooc/blob/main/workbooks/m5_03_tree_hyperparameters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📒 Workbook M5.03 — How big should the tree be?

In the lab we controlled the size of a tree with `max_depth` and
`min_samples_leaf` and chose them by cross-validation. In this workbook
you tune a tree that predicts **whether a wine is good**. The validation
curve will first tell you something that contradicts module 2, and you
will find out why. You will then meet a second way of limiting a tree,
pruning, and a tree that reaches a perfect score for a very bad reason.

**How this workbook works**

- The code is written for you. You do not need to write code, but you need
  to **understand every step**.
- Every `____` is a gap. Fill it in with the value asked for in the text
  **above** the cell. The error `NameError: name '____' is not defined`
  means: a gap is still empty.
- `check(...)` tells you whether your value is right (✅) or stops with a
  hint (❌).
- `commit(...)` records a **prediction** before you see the result. Be
  honest: a wrong prediction costs nothing, it is how you learn.
- Questions marked **🧠 Think** are not checked here. You will need the
  answers for the Moodle quiz and for the discussion at the start of the
  next lab.

In [ ]:
#@title Setup — run this cell first (no need to read it)
import hashlib
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

ANSWERS = {
    "best_tree_all_rows": "11eca344d8aa1471",
    "best_tree_unique_rows": "91d80a4942da5b03",
    "copies_reward": "be45210c91f74a49",
    "leaves_largest_alpha": "6b86b273ff34fce1",
    "most_flexible_min_samples_leaf": "6b86b273ff34fce1",
    "regime_without_pruning": "a1f32f577f69843e",
    "tree_has_learned": "ecec0e8d165d35bd"
}

def _normalize(value):
    if isinstance(value, float) and value.is_integer():
        value = int(value)
    if isinstance(value, float):
        value = round(value, 2)
    if isinstance(value, (list, tuple, set)):
        return ",".join(sorted(str(v).strip().lower() for v in value))
    return str(value).strip().lower()


def _hash(value):
    return hashlib.sha256(_normalize(value).encode()).hexdigest()[:16]


def check(name, value, hint="Re-read the text above this cell and try again."):
    """Stop with a hint unless `value` is the expected answer for `name`."""
    if _hash(value) != ANSWERS[name]:
        raise ValueError(f"❌ {name} = {value!r} is not right. {hint}")
    print(f"✅ {name}")


def check_score(name, value, actual):
    """Stop with a hint unless `value` is `actual` rounded to 2 decimals.

    For results that differ slightly between computers (e.g. k-NN ties).
    """
    if not isinstance(value, (int, float)) or abs(value - round(actual, 2)) > 0.005:
        raise ValueError(
            f"❌ {name} = {value!r} is not right. "
            "Read the score printed above and round it to 2 decimals."
        )
    print(f"✅ {name}")


def commit(prediction, options):
    """Record a prediction before seeing the result."""
    if prediction not in options:
        raise ValueError(f"Choose one of {options} first.")
    print(f"📌 Your prediction: {prediction!r}. Now run the next cell and compare.")

## 1. Data and target

The same red and white wines as before, with their 11 laboratory
measurements. Tasters gave each wine a `quality` mark. We call a wine
**good** when its mark is 7 or more, and we predict that from the
chemistry.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.tree import DecisionTreeClassifier

wine = pd.read_csv(
    "https://raw.githubusercontent.com/marcinsawinski/scikit-learn-mooc/refs/heads/main/datasets/wine_quality.csv"
)
features = [name for name in wine.columns if name not in ("color", "quality")]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
print(f"{len(wine)} wines, share of good wines: {(wine['quality'] >= 7).mean():.2f}")

A model that always answers "not good" is right 80% of the time: this is
the accuracy to beat.

## 2. The validation curve of `max_depth`

We measure the training and testing accuracy for trees of increasing
depth (`None` means no limit).

In [ ]:
def depth_curve(rows):
    """Training and testing accuracy of trees of increasing depth."""
    data, target = rows[features], rows["quality"] >= 7
    curve = {}
    for depth in [1, 2, 3, 5, 8, 12, None]:
        scores = cross_validate(
            DecisionTreeClassifier(max_depth=depth, random_state=0), data, target, cv=cv, return_train_score=True
        )
        curve["no limit" if depth is None else str(depth)] = {
            "training accuracy": scores["train_score"].mean(),
            "testing accuracy": scores["test_score"].mean(),
        }
    return pd.DataFrame(curve).T.rename_axis("max_depth")


curve_all = depth_curve(wine)
curve_all.round(3)

✏️ Which tree has the best **testing** accuracy: a `"shallow"` one
(depth 3 or less) or the `"deepest"` one (no limit)?

In [ ]:
best_tree_all_rows = ____
check("best_tree_all_rows", best_tree_all_rows)

The unlimited tree reaches 100% on the training wines: it has memorised
them. Module 2 taught us that such a model pays for it on new data. Here
it is the **best** on the test folds. Either wine is a special case, or
something is wrong with our test.

## 3. ➕ Going further: the wines that appear twice

Let us look for rows that are exact copies of another row: same 11
measurements, same colour, same mark.

In [ ]:
print(f"{wine.duplicated().sum()} rows are exact copies of an earlier row")
wine[wine.duplicated(keep=False)].sort_values(features).head(4)

More than a thousand rows are repeats (the same wine was probably
entered once per tasting session). When the rows are shuffled into
folds, a wine and its copy often land on **different sides**: the tree
is tested on a wine it has already seen, with its answer.

🔮 **Predict first.** We remove the copies and draw the curve again.
Which tree will now have the best testing accuracy? `"shallow"` or
`"deepest"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["shallow", "deepest"])

In [ ]:
unique_wines = wine.drop_duplicates().reset_index(drop=True)
curve_unique = depth_curve(unique_wines)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
for ax, curve, title in [(axes[0], curve_all, "all rows (with copies)"), (axes[1], curve_unique, "copies removed")]:
    curve.plot(ax=ax, marker="o")
    ax.set(title=title, xlabel="max_depth")
axes[0].set_ylabel("accuracy")
print(f"{len(unique_wines)} distinct wines")
curve_unique.round(3)

✏️ Without the copies, which tree has the best testing accuracy? And
what did the copies reward: `"memorising"` or `"generalising"`?

In [ ]:
best_tree_unique_rows = ____
copies_reward = ____
check("best_tree_unique_rows", best_tree_unique_rows)
check("copies_reward", copies_reward)

The curve now has the shape module 2 promised: the testing accuracy
rises, then **falls** as the tree memorises. With the copies, memorising
was a winning strategy, because part of the "new" wines were old ones.
The 0.85 of section 2 was an illusion, and it would have made us choose
the wrong tree.

This is the trap of the repeated concrete recipes of workbook M2.01, in
its purest form. From here on we work with the distinct wines only.

🧠 **Think.** Removing the copies is right when they are entry errors.
Suppose instead that each row is a real, separate bottle and that many
bottles simply are identical. Would you still remove them? What else
could you do to keep copies on the same side of every split?

## 4. Another brake: `min_samples_leaf`

`max_depth` limits the number of questions. `min_samples_leaf` says that
a question is only allowed if **each of the two new groups keeps at
least that many wines**. It stops the tree from carving out a leaf for
two or three wines.

In [ ]:
data, target = unique_wines[features], unique_wines["quality"] >= 7
rows = {}
for min_samples_leaf in [1, 5, 20, 50, 100, 300, 1000]:
    scores = cross_validate(
        DecisionTreeClassifier(min_samples_leaf=min_samples_leaf, random_state=0),
        data,
        target,
        cv=cv,
        return_train_score=True,
        return_estimator=True,
    )
    rows[min_samples_leaf] = {
        "training accuracy": scores["train_score"].mean(),
        "testing accuracy": scores["test_score"].mean(),
        "leaves": int(np.mean([tree.get_n_leaves() for tree in scores["estimator"]])),
    }
leaf_curve = pd.DataFrame(rows).T.rename_axis("min_samples_leaf").astype({"leaves": int})
leaf_curve.round(3)

✏️ For this hyperparameter, which value gives the **most flexible** tree
(the one that overfits most): `1` or `1000`?

In [ ]:
most_flexible_min_samples_leaf = ____
check("most_flexible_min_samples_leaf", most_flexible_min_samples_leaf, hint="Which value lets the tree make the most leaves?")

As with the number of neighbours in workbook M2.02, a **larger** value
means a **simpler** model. Always ask in which direction a
hyperparameter works before reading its validation curve.

## 5. ➕ Going further: grow first, cut afterwards

`max_depth` and `min_samples_leaf` stop the tree **while it grows**. The
opposite strategy is **pruning**: let the tree grow completely, then cut
back the branches that bring too little.

scikit-learn does this with `ccp_alpha` ("cost-complexity pruning").
Each leaf costs `ccp_alpha`: a branch is kept only if it improves the
purity of the tree by more than what its leaves cost.

🔮 **Predict first.** As `ccp_alpha` grows, the **training** accuracy
will… `"rise"`, `"stay the same"` or `"fall"`?

In [ ]:
my_prediction = ____
commit(my_prediction, ["rise", "stay the same", "fall"])

In [ ]:
rows = {}
for ccp_alpha in [0, 0.0002, 0.0005, 0.001, 0.002, 0.005, 0.05]:
    scores = cross_validate(
        DecisionTreeClassifier(ccp_alpha=ccp_alpha, random_state=0),
        data,
        target,
        cv=cv,
        return_train_score=True,
        return_estimator=True,
    )
    rows[ccp_alpha] = {
        "training accuracy": scores["train_score"].mean(),
        "testing accuracy": scores["test_score"].mean(),
        "leaves": int(np.mean([tree.get_n_leaves() for tree in scores["estimator"]])),
    }
pruning = pd.DataFrame(rows).T.rename_axis("ccp_alpha").astype({"leaves": int})

fig, ax = plt.subplots(figsize=(7, 3.5))
positions = range(len(pruning))
ax.plot(positions, pruning["training accuracy"], "o-", label="training accuracy")
ax.plot(positions, pruning["testing accuracy"], "o-", label="testing accuracy")
ax.set_xticks(positions, [f"{value:g}" for value in pruning.index])
ax.set(xlabel="ccp_alpha", ylabel="accuracy", title="Pruning a fully grown tree")
ax.legend()
pruning.round(3)

✏️ With `ccp_alpha=0` nothing is cut. Is that tree underfitting or
overfitting? And with the largest `ccp_alpha`, how many leaves are left?

In [ ]:
regime_without_pruning = ____
leaves_largest_alpha = ____
check("regime_without_pruning", regime_without_pruning)
check("leaves_largest_alpha", leaves_largest_alpha, hint="Read the last row of the table.")

A tree with a single leaf asks no question at all: it predicts "not
good" for every wine, and its accuracy is the 0.81 of the baseline. The
two ends of this curve are once more a model that memorises and a model
that ignores the data, and `ccp_alpha` is the dial between them.
scikit-learn's example
[Post pruning decision trees with cost complexity pruning](https://scikit-learn.org/1.8/auto_examples/tree/plot_cost_complexity_pruning.html)
shows how to obtain the interesting values of `ccp_alpha` for a given
tree.

🧠 **Think.** The best trees of sections 2, 4 and 5 reach about the same
testing accuracy, a few points above the baseline. What does that tell
you about the choice between the three brakes? And about how much the
chemistry says about the taste?

## 6. ➕ Going further: a perfect tree

Back to the easy question of workbook M5.01: red or white? Somebody
numbered the bottles as they appear in the file and kept that number as
a column, `bottle_id`. Such identifiers travel with real datasets all
the time.

🔮 **Predict first.** We train a tree to tell red from white using
**only** `bottle_id`. Its cross-validated accuracy will be **"about 0.5"**,
**"about 0.75"** (the share of white wines) or **"above 0.99"**?

In [ ]:
my_prediction = ____
commit(my_prediction, ["about 0.5", "about 0.75", "above 0.99"])

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import plot_tree

with_id = wine.assign(bottle_id=np.arange(len(wine)))
id_tree = DecisionTreeClassifier(random_state=0)
id_accuracy = cross_val_score(id_tree, with_id[["bottle_id"]], with_id["color"], cv=cv).mean()
print(f"accuracy with bottle_id alone: {id_accuracy:.4f}")

id_tree.fit(with_id[["bottle_id"]], with_id["color"])
fig, ax = plt.subplots(figsize=(6, 2.5))
_ = plot_tree(id_tree, feature_names=["bottle_id"], class_names=list(id_tree.classes_), filled=True, ax=ax, fontsize=9)

One question, and an almost perfect score, without a single chemical
measurement.

✏️ What has this tree learned? Write `"the chemistry of wine"` or `"the
order of the file"`.

In [ ]:
tree_has_learned = ____
check("tree_has_learned", tree_has_learned)

The file lists all the red wines first, then all the white ones, so the
row number gives the colour away. Cross-validation cannot detect this:
the shuffled test rows carry their number too. The model only fails
where it matters, on **new** bottles: every bottle numbered above 6496
will be declared white, whatever is in it.

A tree makes this easy to catch, because it can be read: one look at
the picture shows a question that makes no sense. Identifiers, row
numbers and dates of entry should be removed before training, or at
least looked at with suspicion when a model finds them useful.

🧠 **Think.** A hospital dataset has a column `patient_id`, given in
order of arrival, and the hospital opened a cancer department last year.
How could a model predicting cancer use `patient_id`? Would its
cross-validated score reveal the problem?

## 🎤 Before the next lab

1. Do the **Moodle quiz** for this workbook (mandatory, not graded).
2. At the start of the next lab, some of you will be asked to present
   (3 minutes, using your notebook on the projector) **one** of:
   - the two validation curves of `max_depth`, with and without the
     copies, and why the copies rewarded the deepest tree;
   - `max_depth`, `min_samples_leaf` and `ccp_alpha`: what each one
     limits and in which direction it works;
   - what a tree with one leaf predicts and why;
   - the `bottle_id` tree: why its perfect score is worthless.